# Assignment 1.1 — AI Stock Market Assistant (LangChain + LangGraph)

**Objective:** an AI assistant that
1. analyses real-time / historical stock data (Yahoo Finance),
2. retrieves and processes financial news with **RAG** (FAISS + HuggingFace embeddings),
3. generates insights with **LLM reasoning**,
4. gives a **simulated BUY / SELL / HOLD** recommendation with justification,
5. runs as a **multi-agent system in LangGraph** where every agent has one job.


## Step 0 — Setup.

In [1]:
import sys
print(sys.version)

import langchain, langgraph, yfinance
print("langchain", langchain.__version__, "| yfinance", yfinance.__version__)
print("Everything is okay!!!")

3.14.8 (v3.14.8:8e6e75d9102, Sep 30 2026, 14:34:30) [Clang 21.0.0 (clang-2100.1.1.101)]
langchain 1.4.5 | yfinance 1.7.0
Everything is okay!!!


## Step 1 — Configuration: API keys, LLM and embeddings

In [2]:
"""
Configuration: loads API keys from .env and creates the LLM + embedding model.
       
"""
import os
import re
import warnings
from functools import lru_cache

# --- Mac (Apple Silicon) safety flags -------------------------------------
# FAISS and PyTorch both ship their own OpenMP library; on macOS this can crash
# with "OMP: Error #15". These flags avoid that and silence tokenizer warnings.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from dotenv import load_dotenv  # noqa: E402

warnings.filterwarnings("ignore")
load_dotenv()

if os.getenv("HUGGINGFACE_API_KEY") and not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = os.getenv("HUGGINGFACE_API_KEY")

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "groq").lower()      
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3.5:2b")   
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")


def get_llm(temperature: float = 0.2):
    """Return a LangChain chat model based on LLM_PROVIDER in the .env file."""
    if LLM_PROVIDER == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)

    if LLM_PROVIDER == "gemini":
       
        from langchain_google_genai import ChatGoogleGenerativeAI
        key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
        if not key:
            raise EnvironmentError("GEMINI_API_KEY is missing. Add it to your .env file.")
        return ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=temperature, google_api_key=key, max_retries=3)

    
    from langchain_groq import ChatGroq
    if not os.getenv("GROQ_API_KEY"):
        raise EnvironmentError("GROQ_API_KEY is missing. Add it to your .env file.")
    kwargs = dict(model=GROQ_MODEL, temperature=temperature, max_retries=5)
    if "gpt-oss" in GROQ_MODEL:
        kwargs["reasoning_effort"] = "low"   # fewer hidden reasoning tokens -> stays inside free-tier limits
    return ChatGroq(**kwargs)


@lru_cache(maxsize=1)
def get_embeddings():
    """Small local embedding model (~90 MB). Runs comfortably on an M1 with 16 GB RAM."""
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )


def strip_think(text) -> str:
    """Some local reasoning models (e.g. qwen3 on Ollama) wrap their thoughts in <think> tags."""
    text = text if isinstance(text, str) else getattr(text, "content", str(text))
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def provider_label() -> str:
    model = {"groq": GROQ_MODEL, "gemini": GEMINI_MODEL, "ollama": OLLAMA_MODEL}.get(LLM_PROVIDER, GROQ_MODEL)
    return f"{LLM_PROVIDER}:{model}"

In [3]:
llm = get_llm()
print("Using:", provider_label())
print(llm.invoke("In one sentence, what is a P/E ratio?").content)

OSError: GROQ_API_KEY is missing. Add it to your .env file.

## Step 2 — Tools: market data & news
Plain Python functions with **yfinance** + **pandas**, then wrapped with the `@tool` decorator (like the *finGraph AI Agent* class notebook) so the LLM can call them.

In [ ]:
"""
Market-data and news tools.

"""
import os
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
from langchain_core.tools import tool



def fetch_price_history(ticker: str, period: str = "6mo") -> pd.DataFrame:
    """Daily OHLCV history from Yahoo Finance."""
    df = yf.Ticker(ticker).history(period=period, interval="1d", auto_adjust=True)
    if df is None or df.empty:
        raise ValueError(f"No price data found for ticker '{ticker}'. Check the symbol (e.g. AAPL, TCS.NS).")
    return df


def _clean(value, digits: int = 2):
    """Round numbers and turn NaN into None so they are JSON friendly."""
    if value is None:
        return None
    try:
        if np.isnan(value):
            return None
    except TypeError:
        return value
    return round(float(value), digits)


def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def compute_indicators(df: pd.DataFrame) -> dict:
    """Technical summary of a price history DataFrame."""
    close = df["Close"]
    sma20 = close.rolling(20).mean()
    sma50 = close.rolling(50).mean()
    rsi = compute_rsi(close)
    daily_ret = close.pct_change().dropna()

    last = close.iloc[-1]
    prev = close.iloc[-2] if len(close) > 1 else last

    def pct_back(days):
        return _clean((last / close.iloc[-days - 1] - 1) * 100) if len(close) > days else None

    summary = {
        "last_close": _clean(last),
        "previous_close": _clean(prev),
        "day_change_pct": _clean((last / prev - 1) * 100),
        "return_1m_pct": pct_back(21),
        "return_3m_pct": pct_back(63),
        "return_period_pct": _clean((last / close.iloc[0] - 1) * 100),
        "period_high": _clean(close.max()),
        "period_low": _clean(close.min()),
        "sma_20": _clean(sma20.iloc[-1]),
        "sma_50": _clean(sma50.iloc[-1]),
        "rsi_14": _clean(rsi.iloc[-1], 1),
        "annual_volatility_pct": _clean(daily_ret.std() * np.sqrt(252) * 100),
        "avg_volume": int(df["Volume"].tail(20).mean()) if "Volume" in df else None,
        "data_points": int(len(close)),
        "as_of": str(close.index[-1].date()),
    }

    # Simple rule-based trend label (gives the LLM a grounded starting point)
    if summary["sma_20"] and summary["sma_50"]:
        if last > summary["sma_20"] > summary["sma_50"]:
            summary["trend"] = "Uptrend (price > SMA20 > SMA50)"
        elif last < summary["sma_20"] < summary["sma_50"]:
            summary["trend"] = "Downtrend (price < SMA20 < SMA50)"
        else:
            summary["trend"] = "Sideways / mixed"
    else:
        summary["trend"] = "Not enough data for SMA50"

    summary["quant_signal"] = quant_signal(summary)
    return summary


def quant_signal(s: dict) -> dict:
    """Transparent score from -3 to +3 using trend, momentum and RSI."""
    score, reasons = 0, []
    if s.get("sma_20") and s.get("sma_50"):
        if s["last_close"] > s["sma_50"]:
            score += 1; reasons.append("Price above 50-day average")
        else:
            score -= 1; reasons.append("Price below 50-day average")
    if s.get("return_1m_pct") is not None:
        if s["return_1m_pct"] > 3:
            score += 1; reasons.append("Positive 1-month momentum")
        elif s["return_1m_pct"] < -3:
            score -= 1; reasons.append("Negative 1-month momentum")
    rsi = s.get("rsi_14")
    if rsi is not None:
        if rsi > 70:
            score -= 1; reasons.append(f"RSI {rsi} = overbought")
        elif rsi < 30:
            score += 1; reasons.append(f"RSI {rsi} = oversold")
    label = "BUY" if score >= 2 else "SELL" if score <= -2 else "HOLD"
    return {"score": score, "label": label, "reasons": reasons}


def chart_data(df: pd.DataFrame, rows: int = 180) -> list:
    """Last N rows of close + moving averages as a list of dicts (safe to keep in graph state)."""
    out = pd.DataFrame({
        "Close": df["Close"],
        "SMA20": df["Close"].rolling(20).mean(),
        "SMA50": df["Close"].rolling(50).mean(),
    }).tail(rows)
    out.index = out.index.strftime("%Y-%m-%d")
    out = out.round(2).reset_index().rename(columns={"index": "Date", "Date": "Date"})
    return out.where(pd.notnull(out), None).to_dict(orient="records")


def fetch_fundamentals(ticker: str) -> dict:
  
    try:
        info = yf.Ticker(ticker).get_info()
    except Exception:
        info = {}
    keys = {
        "longName": "name", "sector": "sector", "industry": "industry", "currency": "currency",
        "marketCap": "market_cap", "trailingPE": "pe_ratio", "forwardPE": "forward_pe",
        "priceToBook": "price_to_book", "dividendYield": "dividend_yield",
        "fiftyTwoWeekHigh": "52w_high", "fiftyTwoWeekLow": "52w_low",
        "debtToEquity": "debt_to_equity", "ebitda": "ebitda", "totalRevenue": "total_revenue",
        "profitMargins": "profit_margin", "recommendationKey": "analyst_consensus",
    }
    data = {new: info.get(old) for old, new in keys.items()}
    data["name"] = data.get("name") or ticker
    return data


def _parse_yf_news(item: dict) -> dict:
    """yfinance has changed its news format over versions - handle both."""
    c = item.get("content", item)
    url = (c.get("canonicalUrl") or {}).get("url") if isinstance(c.get("canonicalUrl"), dict) else c.get("link")
    provider = c.get("provider")
    source = provider.get("displayName") if isinstance(provider, dict) else c.get("publisher", "Yahoo Finance")
    published = c.get("pubDate") or c.get("providerPublishTime")
    if isinstance(published, (int, float)):
        published = datetime.fromtimestamp(published).strftime("%Y-%m-%d")
    return {
        "title": c.get("title", ""),
        "content": c.get("summary") or c.get("description") or c.get("title", ""),
        "source": source or "Yahoo Finance",
        "url": url or "",
        "published": str(published or "")[:10],
    }


def fetch_news(ticker: str, company_name: str = "", max_items: int = 8) -> list:
    """
    Collect recent news about the company.
    Order: Tavily (if TAVILY_API_KEY set) -> Yahoo Finance news -> DuckDuckGo.
    """
    query = f"{company_name or ticker} ({ticker}) stock news"
    items = []

    if os.getenv("TAVILY_API_KEY"):
        try:
            from langchain_tavily import TavilySearch
            res = TavilySearch(max_results=max_items, topic="news").invoke({"query": query})
            for r in (res.get("results", []) if isinstance(res, dict) else []):
                items.append({
                    "title": r.get("title", ""), "content": r.get("content", ""),
                    "source": "Tavily", "url": r.get("url", ""), "published": r.get("published_date", "")[:10] if r.get("published_date") else "",
                })
        except Exception as e:
            print("Tavily search failed:", e)

    if not items:
        try:
            for raw in (yf.Ticker(ticker).news or [])[:max_items]:
                parsed = _parse_yf_news(raw)
                if parsed["title"]:
                    items.append(parsed)
        except Exception as e:
            print("Yahoo news failed:", e)

    if not items:
        try:
            from langchain_community.tools import DuckDuckGoSearchRun
            text = DuckDuckGoSearchRun().invoke(query)
            if text:
                items.append({"title": f"Web search: {query}", "content": text, "source": "DuckDuckGo", "url": "", "published": ""})
        except Exception as e:
            print("DuckDuckGo search failed:", e)

    return items[:max_items]


# ---------------------------------------------------------------------------
# 2. LangChain tools (used by the chat / ReAct agent with ToolNode)
# ---------------------------------------------------------------------------
@tool
def get_stock_price(ticker: str) -> dict:
    """Return the latest closing price, daily change and 1-month return for a stock ticker.
    Example: AAPL, MSFT, TSLA, RELIANCE.NS, TCS.NS, INFY.NS"""
    try:
        s = compute_indicators(fetch_price_history(ticker, "3mo"))
        return {k: s[k] for k in ("last_close", "previous_close", "day_change_pct", "return_1m_pct", "as_of")}
    except Exception as e:
        return {"error": str(e)}


@tool
def get_technical_indicators(ticker: str, period: str = "6mo") -> dict:
    """Return technical indicators (SMA20, SMA50, RSI14, volatility, trend, quant signal) for a ticker.
    period can be 1mo, 3mo, 6mo, 1y, 2y."""
    try:
        return compute_indicators(fetch_price_history(ticker, period))
    except Exception as e:
        return {"error": str(e)}


@tool
def get_company_fundamentals(ticker: str) -> dict:
    """Return company fundamentals such as sector, market cap, P/E ratio, debt to equity, EBITDA and revenue."""
    return fetch_fundamentals(ticker)


@tool
def search_stock_news(ticker: str, company_name: str = "") -> list:
    """Return recent news headlines and summaries about a company/stock."""
    return [{"title": n["title"], "summary": n["content"][:400], "source": n["source"]}
            for n in fetch_news(ticker, company_name, max_items=5)]


STOCK_TOOLS = [get_stock_price, get_technical_indicators, get_company_fundamentals, search_stock_news]

In [ ]:
# Test the helper functions
df = fetch_price_history("AAPL", "6mo")
df.tail()

In [ ]:
indicators = compute_indicators(df)
indicators

In [ ]:
fetch_fundamentals("AAPL")

In [ ]:
news = fetch_news("AAPL", "Apple Inc.")
for n in news:
    print("-", n["published"], "|", n["source"], "|", n["title"])

In [ ]:
# Tool metadata the LLM sees
for t in STOCK_TOOLS:
    print(t.name, "->", t.args)

# Tool binding: does the LLM decide to call a tool?
llm_with_tools = llm.bind_tools(STOCK_TOOLS)
ai = llm_with_tools.invoke("What is the latest price of Infosys on NSE?")
print("Tool calls:", ai.tool_calls)

In [ ]:
# Plot price with moving averages
import pandas as pd
chart = pd.DataFrame(chart_data(df)).set_index("Date")
chart.plot(figsize=(10, 4), title="AAPL close with SMA20 / SMA50");

## Step 3 — RAG over financial news
News → `Document` → `RecursiveCharacterTextSplitter` → `HuggingFaceEmbeddings` → **FAISS** → retriever.

In [ ]:
"""
RAG over financial news.

"""
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter



def news_to_documents(news_items: list) -> list:
    docs = []
    for n in news_items:
        text = f"{n.get('title', '')}\n{n.get('content', '')}".strip()
        if text:
            docs.append(Document(
                page_content=text,
                metadata={"title": n.get("title", ""), "source": n.get("source", ""),
                          "url": n.get("url", ""), "published": n.get("published", "")},
            ))
    return docs


def build_news_vectorstore(news_items: list, embeddings=None):
    """Chunk + embed + index the news in FAISS."""
    docs = news_to_documents(news_items)
    if not docs:
        return None
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    return FAISS.from_documents(chunks, embeddings or get_embeddings())


def format_docs(docs) -> str:
    return "\n\n".join(
        f"[{i}] ({d.metadata.get('source', '')} {d.metadata.get('published', '')}) {d.page_content}"
        for i, d in enumerate(docs, 1)
    )


def retrieve_news_context(news_items: list, question: str, k: int = 4, embeddings=None) -> tuple:
    """Return (context_text, retrieved_docs) for the question."""
    vs = build_news_vectorstore(news_items, embeddings)
    if vs is None:
        return "No recent news found.", []
    retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": k})
    docs = retriever.invoke(question)
    return format_docs(docs), docs

In [ ]:
context, docs = retrieve_news_context(news, "Apple earnings outlook and risks", k=4)
print(context[:1500])

## Step 4 — State & structured-output schemas
`TypedDict` state for LangGraph, Pydantic models for `PydanticOutputParser`.

In [ ]:
"""
State and output schemas.
"""
from typing import Annotated, List, Literal, TypedDict

from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field


# ---------------- Structured LLM outputs ----------------
class QueryIntent(BaseModel):
    ticker: str = Field(description="Yahoo Finance ticker symbol in UPPER CASE, e.g. AAPL, MSFT, TSLA, "
                                    "RELIANCE.NS, TCS.NS. Use an empty string if no company/stock is mentioned.")
    company_name: str = Field(description="Company name if known, else empty string.")
    intent: Literal["full_analysis", "price_check", "news_only", "general"] = Field(
        description="full_analysis = should I buy/sell/hold, analyse, outlook, recommendation; "
                    "price_check = only price / technical numbers; "
                    "news_only = only latest news / sentiment; "
                    "general = finance concept question, chit-chat, or a follow-up question about the "
                    "previous answer (e.g. 'what risks did you mention?').")


class NewsSentiment(BaseModel):
    sentiment: Literal["Positive", "Neutral", "Negative"] = Field(description="Overall news sentiment for the stock")
    score: float = Field(description="Sentiment score between -1 (very negative) and 1 (very positive)")
    key_points: List[str] = Field(description="3 to 5 short key points from the news, each citing [n]")


class Recommendation(BaseModel):
    action: Literal["BUY", "SELL", "HOLD"] = Field(description="Simulated recommendation")
    confidence: int = Field(description="Confidence from 0 to 100")
    time_horizon: str = Field(description="e.g. 'Short term (1-3 months)'")
    justification: List[str] = Field(description="3 to 5 reasons grounded in the given data")
    risks: List[str] = Field(description="2 to 4 key risks")


# ---------------- LangGraph shared state ----------------
class StockState(TypedDict, total=False):
    messages: Annotated[List[AnyMessage], add_messages]   # conversation memory (chat)
    query: str               # user question
    ticker_hint: str         # optional ticker typed in the UI
    period: str              # price history period: 1mo, 3mo, 6mo, 1y, 2y
    ticker: str
    company_name: str
    intent: str
    price_summary: dict
    price_chart: list
    fundamentals: dict
    news_items: list
    news_context: str
    news_sentiment: dict
    analysis: str
    recommendation: dict
    final_report: str
    error: str

## Step 5 — The agents (LangGraph nodes)

In [ ]:
"""
The specialised agents (LangGraph nodes).

    1. Supervisor / Router agent  - understands the question (PydanticOutputParser)
    2. Market Data agent          - yfinance prices, indicators, fundamentals (tools)
    3. News RAG agent             - fetches news, FAISS retrieval, sentiment (RAG + structured output)
    4. Analyst agent              - LLM reasoning over all the evidence
    5. Recommendation agent       - simulated BUY / SELL / HOLD with justification (structured output)
    6. Report node                - assembles the final markdown answer
    7. Chat (ReAct) agent         - general questions, calls tools through ToolNode

Every LLM call uses LangChain:  ChatPromptTemplate | llm | parser   (runnables / LCEL)
"""
import json
import re

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda


DISCLAIMER = ("⚠️ *Educational project only - this is a simulated recommendation generated by an AI, "
              "not financial advice. Do your own research before investing.*")

# Common names -> tickers, used only if the LLM router fails
FALLBACK_TICKERS = {
    "apple": "AAPL", "microsoft": "MSFT", "google": "GOOGL", "alphabet": "GOOGL", "amazon": "AMZN",
    "tesla": "TSLA", "nvidia": "NVDA", "meta": "META", "netflix": "NFLX", "reliance": "RELIANCE.NS",
    "tcs": "TCS.NS", "infosys": "INFY.NS", "hdfc bank": "HDFCBANK.NS", "icici": "ICICIBANK.NS",
    "wipro": "WIPRO.NS", "itc": "ITC.NS", "sbi": "SBIN.NS", "tata motors": "TATAMOTORS.NS",
}


def _compact(d: dict) -> str:
    """Short JSON for prompts (keeps token usage low on the free Groq tier)."""
    return json.dumps({k: v for k, v in (d or {}).items() if v not in (None, "", [])}, default=str)


def _text_chain(prompt, llm):
    return prompt | llm | StrOutputParser() | RunnableLambda(strip_think)


def _structured_chain(prompt, llm, parser):
    return prompt | llm | StrOutputParser() | RunnableLambda(strip_think) | parser


def make_agents(llm):
    """Create all node functions bound to one LLM instance."""

    # ------------------------------------------------------------------ 1
    intent_parser = PydanticOutputParser(pydantic_object=QueryIntent)
    router_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are the supervisor of a stock market assistant. Classify the user's request and extract the stock.\n"
         "Indian NSE stocks use the .NS suffix (e.g. RELIANCE.NS, TCS.NS, INFY.NS). US stocks have no suffix.\n"
         "Use the recent conversation to resolve words like 'it' or 'that company'.\n"
         "Recent conversation:\n{history}\n\n{format_instructions}"),
        ("human", "{query}"),
    ]).partial(format_instructions=intent_parser.get_format_instructions())
    router_chain = _structured_chain(router_prompt, llm, intent_parser)

    def supervisor_agent(state: StockState) -> dict:
        query = state["query"]
        history = "\n".join(
            f"{m.type}: {str(m.content)[:300]}" for m in state.get("messages", [])[-4:]
        ) or "(none)"
        try:
            parsed = router_chain.invoke({"query": query, "history": history})
            ticker, company, intent = parsed.ticker.strip().upper(), parsed.company_name, parsed.intent
        except Exception:
            ticker, company, intent = _fallback_route(query)

        if state.get("ticker_hint"):                     # ticker typed in the UI wins
            ticker = state["ticker_hint"].strip().upper()
            if intent == "general":
                intent = "full_analysis"
        if intent != "general" and not ticker:
            intent = "general"

        # Reset per-run fields (important because the checkpointer remembers old state)
        return {
            "messages": [HumanMessage(content=query)],
            "ticker": ticker, "company_name": company, "intent": intent,
            "period": state.get("period") or "6mo",
            "price_summary": {}, "price_chart": [], "fundamentals": {}, "news_items": [],
            "news_context": "", "news_sentiment": {}, "analysis": "", "recommendation": {},
            "final_report": "", "error": "",
        }

    # ------------------------------------------------------------------ 2
    def market_data_agent(state: StockState) -> dict:
        ticker = state["ticker"]
        try:
            df = fetch_price_history(ticker, state.get("period", "6mo"))
            summary = compute_indicators(df)
            fundamentals = fetch_fundamentals(ticker)
            return {
                "price_summary": summary,
                "price_chart": chart_data(df),
                "fundamentals": fundamentals,
                "company_name": state.get("company_name") or fundamentals.get("name", ticker),
            }
        except Exception as e:
            return {"error": f"Market data error for {ticker}: {e}"}

    # ------------------------------------------------------------------ 3
    sentiment_parser = PydanticOutputParser(pydantic_object=NewsSentiment)
    sentiment_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a financial news analyst. Use ONLY the retrieved news below. "
         "If the news is not about the company, treat it as Neutral.\n\n{format_instructions}"),
        ("human", "Company: {company} ({ticker})\nUser question: {query}\n\nRetrieved news:\n{context}"),
    ]).partial(format_instructions=sentiment_parser.get_format_instructions())
    sentiment_chain = _structured_chain(sentiment_prompt, llm, sentiment_parser)

    def news_rag_agent(state: StockState) -> dict:
        ticker = state["ticker"]
        company = state.get("company_name") or ticker
        news = fetch_news(ticker, company)
        rag_question = f"{company} {ticker} earnings outlook growth risks analyst rating {state['query']}"
        context, _ = retrieve_news_context(news, rag_question, k=4)
        try:
            sentiment = sentiment_chain.invoke(
                {"company": company, "ticker": ticker, "query": state["query"], "context": context}
            ).model_dump()
        except Exception as e:
            sentiment = {"sentiment": "Neutral", "score": 0.0, "key_points": [f"Sentiment could not be parsed: {e}"]}
        return {"news_items": news, "news_context": context, "news_sentiment": sentiment}

    # ------------------------------------------------------------------ 4
    analyst_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a senior equity research analyst with 15 years of experience. Write a concise, data-driven "
         "analysis in Markdown with these headings: ### Price Trend & Momentum, ### Valuation & Fundamentals, "
         "### News Impact, ### Key Insight. Quote the actual numbers. Do not invent data. Max 250 words."),
        ("human",
         "Stock: {company} ({ticker})\nUser question: {query}\n\nTechnical summary: {price}\n\n"
         "Fundamentals: {fundamentals}\n\nNews sentiment: {sentiment}"),
    ])
    analyst_chain = _text_chain(analyst_prompt, llm)

    def analyst_agent(state: StockState) -> dict:
        analysis = analyst_chain.invoke({
            "company": state.get("company_name", ""), "ticker": state["ticker"], "query": state["query"],
            "price": _compact(state.get("price_summary")), "fundamentals": _compact(state.get("fundamentals")),
            "sentiment": _compact(state.get("news_sentiment")),
        })
        return {"analysis": analysis}

    # ------------------------------------------------------------------ 5
    rec_parser = PydanticOutputParser(pydantic_object=Recommendation)
    rec_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a portfolio advisor creating a SIMULATED recommendation for an educational project. "
         "Decide BUY, SELL or HOLD using the analyst report, the rule-based quant signal and news sentiment. "
         "If evidence conflicts, prefer HOLD and lower confidence. Ground every reason in the data.\n\n"
         "{format_instructions}"),
        ("human",
         "Stock: {ticker}\nQuant signal: {signal}\nNews sentiment: {sentiment}\n\nAnalyst report:\n{analysis}"),
    ]).partial(format_instructions=rec_parser.get_format_instructions())
    rec_chain = _structured_chain(rec_prompt, llm, rec_parser)

    def recommendation_agent(state: StockState) -> dict:
        signal = state.get("price_summary", {}).get("quant_signal", {})
        try:
            rec = rec_chain.invoke({
                "ticker": state["ticker"], "signal": _compact(signal),
                "sentiment": _compact(state.get("news_sentiment")), "analysis": state.get("analysis", ""),
            }).model_dump()
        except Exception:
            # Fallback: use the transparent quant signal
            rec = {"action": signal.get("label", "HOLD"), "confidence": 50, "time_horizon": "Short term (1-3 months)",
                   "justification": signal.get("reasons", ["Insufficient data"]),
                   "risks": ["LLM output could not be parsed - rule-based signal used"]}
        return {"recommendation": rec}

    # ------------------------------------------------------------------ 6
    def report_node(state: StockState) -> dict:
        report = build_report(state)
        return {"final_report": report, "messages": [AIMessage(content=report)]}

    # ------------------------------------------------------------------ 7
    llm_with_tools = llm.bind_tools(STOCK_TOOLS)
    chat_system = (
        "You are a helpful stock market assistant. Use the tools for any live price, indicator, fundamental "
        "or news question instead of guessing. Explain finance concepts simply. "
        "Never give real financial advice; say recommendations are simulated."
    )

    def chat_agent(state: StockState) -> dict:
        msgs = state["messages"]
        # Keep only the current turn (last HumanMessage onward) + a short text summary of earlier turns
        last_human = max(i for i, m in enumerate(msgs) if isinstance(m, HumanMessage))
        earlier = "\n".join(f"{m.type}: {str(m.content)[:300]}" for m in msgs[max(0, last_human - 4):last_human])
        system = SystemMessage(content=chat_system + (f"\n\nEarlier conversation:\n{earlier}" if earlier else ""))
        response = llm_with_tools.invoke([system] + msgs[last_human:])
        update = {"messages": [response]}
        if not getattr(response, "tool_calls", None):
            update["final_report"] = strip_think(response.content)
        return update

    return {
        "supervisor": supervisor_agent,
        "market_data": market_data_agent,
        "news_rag": news_rag_agent,
        "analyst": analyst_agent,
        "recommender": recommendation_agent,
        "report": report_node,
        "chat": chat_agent,
    }


def _fallback_route(query: str):
    """Very small keyword router used only if the LLM output cannot be parsed."""
    q = query.lower()
    ticker, company = "", ""
    for name, sym in FALLBACK_TICKERS.items():
        if name in q:
            ticker, company = sym, name.title()
            break
    if not ticker:
        m = re.search(r"\b([A-Z]{2,10}(?:\.NS|\.BO)?)\b", query)
        ticker = m.group(1) if m else ""
    if "news" in q:
        intent = "news_only"
    elif "price" in q and "buy" not in q and "sell" not in q:
        intent = "price_check"
    else:
        intent = "full_analysis" if ticker else "general"
    return ticker, company, intent


def _fmt(v, suffix=""):
    if v is None:
        return "n/a"
    if isinstance(v, (int, float)) and abs(v) >= 1e9:
        return f"{v/1e9:,.2f} B{suffix}"
    if isinstance(v, float):
        return f"{v:,.2f}{suffix}"
    return f"{v}{suffix}"


def build_report(state: dict) -> str:
    """Markdown report assembled from the state (no LLM call)."""
    t = state.get("ticker", "")
    name = state.get("company_name") or t
    if state.get("error"):
        return f"❌ {state['error']}\n\nTip: use Yahoo Finance symbols, e.g. `AAPL`, `MSFT`, `RELIANCE.NS`, `TCS.NS`."

    lines = [f"## 📈 {name} ({t})"]
    p, f = state.get("price_summary") or {}, state.get("fundamentals") or {}
    if p:
        cur = f.get("currency") or ""
        lines += [
            "### Market Data",
            f"- **Last close:** {_fmt(p.get('last_close'))} {cur} ({_fmt(p.get('day_change_pct'), '%')} day change) — as of {p.get('as_of')}",
            f"- **Returns:** 1M {_fmt(p.get('return_1m_pct'), '%')}, 3M {_fmt(p.get('return_3m_pct'), '%')}",
            f"- **SMA20 / SMA50:** {_fmt(p.get('sma_20'))} / {_fmt(p.get('sma_50'))} → {p.get('trend')}",
            f"- **RSI(14):** {_fmt(p.get('rsi_14'))}  |  **Volatility (annual):** {_fmt(p.get('annual_volatility_pct'), '%')}",
            f"- **P/E:** {_fmt(f.get('pe_ratio'))}  |  **Market cap:** {_fmt(f.get('market_cap'))}  |  **Sector:** {f.get('sector') or 'n/a'}",
            f"- **Quant signal:** {p.get('quant_signal', {}).get('label')} (score {p.get('quant_signal', {}).get('score')})",
        ]
    s = state.get("news_sentiment") or {}
    if s:
        lines += [f"### 📰 News Sentiment: {s.get('sentiment')} (score {s.get('score')})"]
        lines += [f"- {kp}" for kp in s.get("key_points", [])]
        news = state.get("news_items") or []
        if news:
            lines += ["", "**Sources:**"] + [
                f"{i}. [{n['title']}]({n['url']}) — {n['source']}" if n.get("url") else f"{i}. {n['title']} — {n['source']}"
                for i, n in enumerate(news[:5], 1)
            ]
    if state.get("analysis"):
        lines += ["### 🧠 Analyst Insights", state["analysis"]]
    r = state.get("recommendation") or {}
    if r:
        lines += [f"### 🎯 Simulated Recommendation: **{r.get('action')}** (confidence {r.get('confidence')}%)",
                  f"*Horizon: {r.get('time_horizon')}*", "**Why:**"]
        lines += [f"- {x}" for x in r.get("justification", [])]
        lines += ["**Risks:**"] + [f"- {x}" for x in r.get("risks", [])]
    lines += ["", DISCLAIMER]
    return "\n".join(lines)

In [ ]:
# Test one agent on its own
agents = make_agents(llm)
agents["supervisor"]({"query": "Should I invest in Reliance Industries now?", "messages": []})

## Step 6 — Build the LangGraph workflow
Nodes, normal edges and **conditional edges**, compiled with an `InMemorySaver` checkpointer for memory.

In [ ]:

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition



# ---------------- Conditional-edge router functions ----------------
def route_after_supervisor(state: StockState) -> str:
    intent = state.get("intent")
    if intent in ("full_analysis", "price_check"):
        return "market_data"
    if intent == "news_only":
        return "news_rag"
    return "chat"


def route_after_market(state: StockState) -> str:
    if state.get("error"):
        return "report"            # invalid ticker / no data -> skip the LLM agents
    if state.get("intent") == "price_check":
        return "report"
    return "news_rag"


def route_after_news(state: StockState) -> str:
    return "report" if state.get("intent") == "news_only" else "analyst"


def build_graph(llm=None, with_memory: bool = True):
    llm = llm or get_llm()
    agents = make_agents(llm)

    builder = StateGraph(StockState)

    # Nodes (each node = one specialised agent)
    builder.add_node("supervisor", agents["supervisor"])
    builder.add_node("market_data", agents["market_data"])
    builder.add_node("news_rag", agents["news_rag"])
    builder.add_node("analyst", agents["analyst"])
    builder.add_node("recommender", agents["recommender"])
    builder.add_node("report", agents["report"])
    builder.add_node("chat", agents["chat"])
    builder.add_node("tools", ToolNode(STOCK_TOOLS))

    # Edges
    builder.add_edge(START, "supervisor")
    builder.add_conditional_edges("supervisor", route_after_supervisor,
                                  {"market_data": "market_data", "news_rag": "news_rag", "chat": "chat"})
    builder.add_conditional_edges("market_data", route_after_market,
                                  {"news_rag": "news_rag", "report": "report"})
    builder.add_conditional_edges("news_rag", route_after_news,
                                  {"analyst": "analyst", "report": "report"})
    builder.add_edge("analyst", "recommender")
    builder.add_edge("recommender", "report")
    builder.add_edge("report", END)

    # ReAct loop for general chat questions (same pattern as the Day 16 chatbot)
    builder.add_conditional_edges("chat", tools_condition)   # -> "tools" or END
    builder.add_edge("tools", "chat")

    return builder.compile(checkpointer=InMemorySaver() if with_memory else None,
                           name="stock_market_multi_agent")


def ask(graph, query: str, thread_id: str = "1", ticker_hint: str = "", period: str = "6mo") -> dict:
    """Convenience helper: run one question through the graph."""
    config = {"configurable": {"thread_id": thread_id}}
    return graph.invoke({"query": query, "ticker_hint": ticker_hint, "period": period}, config=config)

In [ ]:
graph = build_graph(llm)

from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:                       # needs internet for mermaid.ink
    print(graph.get_graph().draw_mermaid())

## Step 7 — Run the assistant
### 7.1 Full analysis (supervisor → market_data → news_rag → analyst → recommender → report)

In [ ]:
from IPython.display import Markdown

result = ask(graph, "Should I buy Apple stock right now?", thread_id="demo-1")
print("Route:", result["intent"], "| Ticker:", result["ticker"])
Markdown(result["final_report"])

### 7.2 Indian stock, price check only (conditional edge skips news & LLM agents)

In [ ]:
result = ask(graph, "What is the current price and RSI of TCS?", thread_id="demo-2")
print("Route:", result["intent"], "| Ticker:", result["ticker"])
Markdown(result["final_report"])

### 7.3 News only (conditional edge goes news_rag → report)

In [ ]:
result = ask(graph, "What is the latest news sentiment on Nvidia?", thread_id="demo-3")
Markdown(result["final_report"])

### 7.4 Error handling — invalid ticker (conditional edge market_data → report)

In [ ]:
result = ask(graph, "Analyse the stock XYZ123QQ", thread_id="demo-4", ticker_hint="XYZ123QQ")
Markdown(result["final_report"])

### 7.5 General question → ReAct chat agent with tools (`tools_condition`)

In [ ]:
result = ask(graph, "Explain what RSI means and tell me Microsoft's current RSI.", thread_id="demo-5")
for m in result["messages"][-4:]:
    m.pretty_print()

### 7.6 Memory — follow-up question in the same thread

In [ ]:
ask(graph, "Give me a full analysis of Infosys", thread_id="demo-6")
follow_up = ask(graph, "What were the main risks you mentioned for it?", thread_id="demo-6")
print(follow_up["final_report"])

### 7.7 Streaming — watch each agent finish

In [ ]:
config = {"configurable": {"thread_id": "demo-7"}}
for update in graph.stream({"query": "Analyse HDFC Bank", "ticker_hint": "", "period": "6mo"},
                           config=config, stream_mode="updates"):
    for node, value in update.items():
        print(f"✅ {node:12s} -> updated: {list(value.keys()) if value else []}")


## Conclusion & limitations
- Multi-agent design keeps each prompt small and focused; conditional edges skip agents that are not needed (faster & cheaper).
- Recommendations combine a transparent rule-based **quant signal** with LLM reasoning and news sentiment; if the LLM output cannot be parsed the system falls back to the rule-based signal.
- Limitations: free news sources are limited, yfinance data can be delayed, LLM outputs can vary — **not financial advice**.
- Future work: backtesting the signal, more indicators (MACD, Bollinger), portfolio-level analysis, LangSmith tracing.